In [12]:
from pydantic import BaseModel
import json

# create class for data validation
class CustomerRecord(BaseModel):
    customer_id : str
    first_name  : str
    last_name   : str
    email       : str
    city        : str
    state       : str

customer =  CustomerRecord(
    customer_id = '1001',
    first_name = 'siva',
    last_name = 'sankar',
    email = 'siva@gmail.com',
    city = 'Hyderabad',
    state = 'A.P' 
)

print(repr(customer))
print(str(customer))
print('customer_id type:', type(customer.customer_id))

CustomerRecord(customer_id='1001', first_name='siva', last_name='sankar', email='siva@gmail.com', city='Hyderabad', state='A.P')
customer_id='1001' first_name='siva' last_name='sankar' email='siva@gmail.com' city='Hyderabad' state='A.P'
customer_id type: <class 'str'>


In [43]:
from pydantic import BaseModel, Field, ValidationError, EmailStr
from typing import Optional, Literal

# create class for data validation
class CustomerRecord(BaseModel):
    customer_id : int = Field(gt = 0, description ='Must be positive value')
    first_name  : str = Field(min_length = 1 ,max_length = 50)
    last_name   : str = Field(min_length = 1 ,max_length = 50)
    email       : EmailStr
    city        : str 
    state       : str = Field(max_length = 3)

    @property
    def full_name(self):
        return f' {self.first_name} {self.last_name}'

    

customer =  CustomerRecord(
    customer_id = 1001,
    first_name = 'siva',
    last_name = 'sankar',
    email = 'siva@gmail.com',
    city = 'Hyderabad',
    state = 'A.P')

print(customer.full_name, '|', customer.city)

 siva sankar | Hyderabad


In [44]:
# Invalid — triggers ValidationError with clear field-level messages
try:
    bad = CustomerRecord(
        customer_id = -5,     # violates gt=0
        first_name  = '',     # violates min_length=1
        last_name   = 'Akella',
        email       = 'Not_an_email', 
        city        = 'Rajahmundry',
        state       = 'AndhraPradesh',  # violates max_length=2
    )
except ValidationError as e:
    for err in e.errors():
        print(f' {err['loc']} - >{err['msg']}')
    
        
        
    
       
      
    

 ('customer_id',) - >Input should be greater than 0
 ('first_name',) - >String should have at least 1 character
 ('email',) - >value is not a valid email address: An email address must have an @-sign.
 ('state',) - >String should have at most 3 characters


In [55]:
from pydantic import BaseModel, Field, ValidationError, EmailStr
from typing import Optional, Literal

# create class for data validation
class CustomerRecord(BaseModel):
    customer_id : int = Field(gt = 0, description ='Must be positive value')
    name  : str = Field(min_length = 1 ,max_length = 50)
    email       : EmailStr
    city        : str 
    state       : str = Field(default = 'A.P')

    @property
    def full_name(self):
        return f' {self.first_name} {self.last_name}'

raw_records = [
    {'customer_id': 1001, 'name': 'Praneeth', 'email': 'praneeth@example.com', 'city': 'Hyderabad'},
    {'customer_id': 1002, 'name': 'Ravi',     'email': 'ravi@example.com',     'city': 'Bangalore'},
    {'customer_id': 'abc', 'name': 'Kumar',   'email': 'kumar@example.com',    'city': 'Chennai'},  # ❌ invalid customer_id
]

Valid_records = []
Invalid_records = []

for row in raw_records:
    try:
        customer = CustomerRecord(**row)
        Valid_records.append(raw)
    except Exception as e:
        Invalid_records.append({'record:', raw, 'error:', str(e)})

print(f'valid_records : {len(Valid_records)} ,Inavlid_records:{len(Invalid_records)}')
        

valid_records : 2 ,Inavlid_records:1


In [58]:
import psycopg2
from psycopg2 import OperationalError

def test_connection():
    try:
        conn = psycopg2.connect(
            host     = "localhost",
            port     = "5432",
            database = "postgres",
            user     = "postgres",
            password = "admin1234",
            connect_timeout = 5
        )
        print("✅ Connection established successfully!")
        
        cur = conn.cursor()
        cur.execute("SELECT * from public.dept;")
        db, user, version = cur.fetchone()
        print(f"Database : {db}")
        print(f"User     : {user}")
        print(f"Version  : {version}")
        
        cur.close()
        conn.close()
        print("✅ Connection closed properly.")
        
    except OperationalError as e:
        print(f"❌ Connection failed: {e}")

test_connection()

✅ Connection established successfully!
Database : postgres
User     : postgres
Version  : PostgreSQL 18.4 (Homebrew) on aarch64-apple-darwin25.4.0, compiled by Apple clang version 21.0.0 (clang-2100.0.123.102), 64-bit
✅ Connection closed properly.


In [79]:
import psycopg2
from psycopg2 import OperationalError
from pydantic import BaseModel, ValidationError


class Customer(BaseModel):                    # ✅ capitalized class name
    customer_id:   int
    customer_name: str
    phone_number:  int
    city:          str
    country:       str


def get_connection():
    """Create and return a database connection"""
    try:
        conn = psycopg2.connect(
            host            = 'localhost',
            port            = 5432,
            database        = 'postgres',
            user            = 'postgres',
            password        = 'admin1234',
            connect_timeout = 5
        )
        print("✅ Connection established successfully!")
        return conn

    except OperationalError as e:              # ✅ fixed typo
        print(f"❌ Connection failed: {e}")
        return None


# ✅ everything below is OUTSIDE the function — correct indentation
conn = get_connection()

if conn:
    cur = conn.cursor()
    cur.execute("SELECT * FROM public.customers;")

    columns = ['customer_id', 'customer_name', 'phone_number', 'city', 'country']
    rows    = cur.fetchall()

    valid_customers  = []                      # ✅ correct indentation
    invalid_records  = []

    for row in rows:
        record = dict(zip(columns, row))
        try:
            customer = Customer(**record)      # ✅ matches class name now
            valid_customers.append(customer)
        except ValidationError as e:
            invalid_records.append({'record': record, 'error': e.errors()})

    print(f'Valid: {len(valid_customers)}, Invalid: {len(invalid_records)}')

    for cust in valid_customers:
        print(cust)                            # ✅ only print inside loop

    cur.close()                                # ✅ close AFTER loop, outside it
    conn.close()
            
            
        


    


✅ Connection established successfully!
Valid: 0, Invalid: 0


In [45]:
import psycopg2
from pydantic import BaseModel,ValidationError,Field
from psycopg2 import OperationalError

class Interview(BaseModel):
    id : int = Field(ge = 1 , le =10)
    topic : str
    status : str
    


def test_connection():
    try:
        conn = psycopg2.connect(
        host = 'localhost',
        port = 5432,
        database = 'postgres',
        user = 'postgres',
        password = 'admin1234',
        connect_timeout =5
        )
        print(f'connection successful !')
        return conn
    except OperationalError as e:
        print('connection could not be established')
        return None
    
    
    


    
        
conn = test_connection()

if conn:
    cur = conn.cursor()
    cur.execute('select * from "Neethu".interview;')

    valid_records =[]
    Invalid_records =[]
    
    columns = ['id', 'topic', 'status']
    rows = cur.fetchall()
    
    for row in rows:
        
        record = dict(zip(columns,row))
        try:
            customer = Interview(**record)
            valid_records.append(customer)
        except ValidationError as e:
            Invalid_records.append({'record': record ,'error' : e.errors()})
            
    
    print(f'Valid: {len(valid_records)}, Invalid: {len(Invalid_records)}')
    print()
    print('valid_custoemrs are:-')
    for r in valid_records:
        print(r)
    print()
    print('Invalid_customers are:-')
    for x in Invalid_records:
        print(x)
         
         
    
    cur.close()
    conn.close()


     
         
        
    




connection successful !
Valid: 3, Invalid: 1

valid_custoemrs are:-
id=1 topic='Database connection' status='Completed'
id=2 topic='Writing queries' status='In Progress'
id=3 topic='Joins and Aggregations' status='Not started'

Invalid_customers are:-
{'record': {'id': 12, 'topic': 'Delivery', 'status': 'not completed'}, 'error': [{'type': 'less_than_equal', 'loc': ('id',), 'msg': 'Input should be less than or equal to 10', 'input': 12, 'ctx': {'le': 10}, 'url': 'https://errors.pydantic.dev/2.8/v/less_than_equal'}]}


In [51]:
rows = {
    'product_id': '101',       # str — Pydantic converts to int
    'product_name': 'Classic Monitor',
    'category': 'Electronics',
    'price': '205.21',         # str — Pydantic converts to float
    'stock_quantity': '238',
}

class ProductReview(BaseModel):
    product_id : int
    product_name: str
    category : str
    price :float
    stock_quantity : int

product_1 = ProductReview(**rows)
print(product_1)
product_2 = ProductReview.model_validate(rows)
print(product_2)
print('price type is :', type(product_1.price).__name__)

product_id=101 product_name='Classic Monitor' category='Electronics' price=205.21 stock_quantity=238
product_id=101 product_name='Classic Monitor' category='Electronics' price=205.21 stock_quantity=238
price type is : float


In [1]:
product = ProductRecord(product_id=101, product_name='Classic Monitor',
                        category='Electronics', price=205.21, stock_quantity=238)

p = product.model_dump()
Print(type(p).__name__)

NameError: name 'ProductRecord' is not defined

In [2]:
# model_validate() — validate a Python dict
# Use this when you have a dict (from csv.DictReader, database row, etc.)
row = {
    'product_id': '101',       # str — Pydantic converts to int
    'product_name': 'Classic Monitor',
    'category': 'Electronics',
    'price': '205.21',         # str — Pydantic converts to float
    'stock_quantity': '238',
}

product = ProductRecord(**row)
print(product)
product = ProductRecord.model_validate(row)
print(product)
print('price type  :', type(product.price).__name__)    # float (was str)
print('stock type  :', type(product.stock_quantity).__name__)  # int (was str)


NameError: name 'ProductRecord' is not defined

In [ ]:
import psycopg2
from psycopg2 import BaseModel,Literal
class TriageOutcome(BaseModel):
    category : Literal['TRACK_ORDER', 'CANCEL', 'REFUND', 'GENERAL']
    confidence : Literal['high', 'medium', 'low']
    reason     : str = Field(min_length=5)

    model_config 